In [10]:
# Standardbibliotheken
import os
import random
import time
from pathlib import Path

# Numerische Berechnungen
import numpy as np
import pandas as pd

# Visualisierung
import matplotlib.pyplot as plt

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

# Dataset & DataLoader
from torch.utils.data import Dataset, DataLoader, random_split

# Metriken
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Reproduzierbarkeit
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Device auswählen
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch Version:", torch.__version__)
print("Device:", device)

PyTorch Version: 2.12.0+cpu
Device: cpu


In [ ]:
import torch.nn as nn

def build_mlp(input_size, hidden_layers, output_size):
    layers = []

    sizes = [input_size] + hidden_layers + [output_size]

    for i in range(len(sizes) - 1):
        layers.append(nn.Linear(sizes[i], sizes[i+1]))

        # activation after hidden layers only
        if i < len(hidden_layers):
            layers.append(nn.SiLU())#nehme swish hier als pytorch uiolt in oder from tensorflow. lerne paramtr wenn es geht! 
        else:
            layers.append(nn.Identity())
    return nn.Sequential(*layers)

In [12]:
model = build_mlp(
    input_size=20,
    hidden_layers=[128, 64, 32],
    output_size=3
)

print(model)

Sequential(
  (0): Linear(in_features=20, out_features=128, bias=True)
  (1): SiLU()
  (2): Linear(in_features=128, out_features=64, bias=True)
  (3): SiLU()
  (4): Linear(in_features=64, out_features=32, bias=True)
  (5): SiLU()
  (6): Linear(in_features=32, out_features=3, bias=True)
  (7): Identity()
)


In [ ]:
from pathlib import Path
import pandas as pd
import re


def create_nn_dataset(
        op_name,
        target_header,
        target_type="grid"
):
    """
    target_type:
        "grid"
        "fmu"

    Beispiele:

    create_nn_dataset(
        "OP02",
        "cc_15",
        "grid"
    )

    create_nn_dataset(
        "OP02",
        "bc_SOC Monitor",
        "fmu"
    )
    """

    root = Path(".")

    op_folder = root / "storeOPs" / op_name

    if not op_folder.exists():
        raise FileNotFoundError(op_folder)

    # --------------------------------------------------
    # INPUTS
    # --------------------------------------------------

    input_file = next(op_folder.glob("*Input*.csv"))

    input_df = pd.read_csv(input_file)

    input_values = input_df.iloc[0].to_dict()

    # --------------------------------------------------
    # GRID OUTPUT
    # --------------------------------------------------

    if target_type.lower() == "grid":

        tgrid_file = None

        for f in op_folder.glob("*T_grid*.csv"):

            cols = pd.read_csv(f, nrows=1).columns

            if any(target_header in c for c in cols):
                tgrid_file = f
                break

        if tgrid_file is None:
            raise ValueError(
                f"{target_header} nicht gefunden"
            )

        df = pd.read_csv(tgrid_file)

        time_col = df.columns[0]

        target_col = None

        for c in df.columns:
            if target_header in c:
                target_col = c
                break

        monitor_idx = int(
            re.findall(r"(\d+)", target_col)[-1]
        )

        # passende Coordinate-Datei finden

        if "cc" in target_col.lower():

            coord_file = (
                root
                / "coordinates"
                / "Coordinates - Grid Cell Center.csv"
            )

        elif "jr1c" in target_col.lower():

            coord_file = (
                root
                / "coordinates"
                / "Coordinates - Grid JR1 Center.csv"
            )

        else:

            coord_file = (
                root
                / "coordinates"
                / "Coordinates - Grid Gehäusewand.csv"
            )

        coord_df = pd.read_csv(coord_file)

        xyz = coord_df.iloc[monitor_idx - 1]

        result = pd.DataFrame()

        result["t"] = df[time_col]

        result["x"] = xyz.iloc[0]
        result["y"] = xyz.iloc[1]
        result["z"] = xyz.iloc[2]

        for key, value in input_values.items():
            result[key] = value

        result["target"] = df[target_col]

        return result

    # --------------------------------------------------
    # FMU OUTPUT
    # --------------------------------------------------

    elif target_type.lower() == "fmu":

        fmu_files = list(op_folder.glob("*FMU*.csv"))

        found_df = None

        found_col = None

        for file in fmu_files:

            tmp = pd.read_csv(file)

            if target_header in tmp.columns:

                found_df = tmp
                found_col = target_header
                break

        if found_df is None:
            raise ValueError(
                f"{target_header} nicht gefunden"
            )

        result = pd.DataFrame()

        result["t"] = found_df.iloc[:, 0]

        result["x"] = 0
        result["y"] = 0
        result["z"] = 0

        for key, value in input_values.items():
            result[key] = value

        result["target"] = found_df[found_col]

        return result

    else:

        raise ValueError(
            "target_type muss 'grid' oder 'fmu' sein"
        ) 
    


 
